In [15]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import ipywidgets as widgets
from IPython.display import display, clear_output

# ===============================
# LOAD DATA
# ===============================

df = pd.read_csv("Ateek_Ford.csv")

df = df.drop(columns=["S.NO"], errors="ignore")

df["Mileage"] = df["Mileage.Done"].str.replace(",", "").astype(float)
df["Price"] = df["Price.in.USD"].str.replace(r"[^\d.]", "", regex=True).astype(float)

df = df.drop(columns=["Mileage.Done", "Price.in.USD"])

# Feature Engineering
CURRENT_YEAR = 2026
df["Vehicle_Age"] = np.maximum(CURRENT_YEAR - df["Year"], 1)
df["Mileage_per_Year"] = df["Mileage"] / df["Vehicle_Age"]
df["MPG"] = 25

# Simulate Maintenance (temporary)
np.random.seed(42)

df["Annual_Repair_Cost"] = (
    500
    + 180 * df["Vehicle_Age"]
    + 120 * (df["Mileage_per_Year"] / 10000)
    + np.where(df["Drive.Train"] == "All-Wheel Drive", 250, 0)
    + np.random.normal(0, 300, len(df))
)

df["Annual_Repair_Cost"] = df["Annual_Repair_Cost"].clip(lower=200)

# Encode categorical
df_encoded = pd.get_dummies(
    df,
    columns=["MODEL", "Trim", "Color", "Drive.Train", "Location..State."],
    drop_first=True
)

# ===============================
# TRAIN MODEL
# ===============================

X = df_encoded.drop(columns=["Annual_Repair_Cost"])
y = df_encoded["Annual_Repair_Cost"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

model = RandomForestRegressor(n_estimators=500, random_state=123)
model.fit(X_train, y_train)

print("Model trained successfully.")
print("R²:", r2_score(y_test, model.predict(X_test)))
print("RMSE:", np.sqrt(mean_squared_error(y_test, model.predict(X_test))))

# ===============================
# UI SECTION
# ===============================

ownership_slider = widgets.IntSlider(value=5, min=1, max=10, description="Years")
miles_slider = widgets.IntSlider(value=12000, min=8000, max=25000, description="Annual Miles")
fuel_slider = widgets.FloatSlider(value=3.5, min=2.0, max=6.0, step=0.1, description="Fuel Price")

button = widgets.Button(description="Calculate TCO")

output = widgets.Output()

def calculate_tco(b):
    with output:
        clear_output()
        
        years = ownership_slider.value
        annual_miles = miles_slider.value
        fuel_price = fuel_slider.value
        
        df_encoded["Predicted_Repair"] = model.predict(X)
        
        df_encoded["Fuel_Cost"] = (
            (annual_miles / df_encoded["MPG"]) * fuel_price * years
        )
        
        df_encoded["Maintenance_Total"] = (
            df_encoded["Predicted_Repair"] * years
        )
        
        df_encoded["Resale_Value"] = df_encoded["Price"] * 0.5
        
        df_encoded["TCO"] = (
            df_encoded["Price"]
            + df_encoded["Maintenance_Total"]
            + df_encoded["Fuel_Cost"]
            - df_encoded["Resale_Value"]
        )
        
        ranked = df_encoded.sort_values("TCO")[["Price","Predicted_Repair","TCO"]]
        
        print("Top 5 Lowest TCO Vehicles:")
        display(ranked.head())

button.on_click(calculate_tco)

display(ownership_slider, miles_slider, fuel_slider, button, output)


KeyError: 'Mileage.Done'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv("car_data.csv")

# Create Vehicle_Age (static assumption for now)
CURRENT_YEAR = 2026
df["Vehicle_Age"] = CURRENT_YEAR - df["Year"]

# Plot
sns.scatterplot(x="Vehicle_Age", y="Annual_Repair_Cost", data=df)
plt.title("Repair Cost vs Vehicle Age")
plt.xlabel("Vehicle Age (Years)")
plt.ylabel("Annual Repair Cost ($)")
plt.show()
